# Tutorial 3: Multilingual Generation

Generating synthetic data for low-resource languages is one of the strongest use cases for `synthetictext`. This tutorial covers:

1. Configuring multiple languages with `LanguageConfig`
2. Direct generation in non-English languages
3. Paraphrasing existing training data
4. Backtranslation (translate out and back to augment)
5. Pivot translation (generate in a high-resource language, then translate)
6. Generating across all configured languages at once

In [ ]:
import os
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"

## Setting Up a Multilingual Task

Define your task once -- it applies to all languages.

In [ ]:
from synthetictext import TaskSpec, LanguageConfig, SyntheticDataGenerator

task = TaskSpec(
    name="Hate Speech Detection",
    labels={0: "not-hateful", 1: "hateful"},
    description=(
        "Classify social media posts as containing hate speech or not. "
        "Hate speech targets individuals or groups based on protected "
        "characteristics such as race, religion, gender, or nationality."
    ),
    label_descriptions={
        0: (
            "A post that may express opinions or disagreement but does not "
            "attack, threaten, or dehumanize any group or individual based "
            "on protected characteristics."
        ),
        1: (
            "A post containing slurs, threats, dehumanization, calls for "
            "violence, or discriminatory language targeting a group based "
            "on race, religion, ethnicity, gender, sexuality, or nationality."
        ),
    },
    text_domain="social media post",
    word_count_range=(15, 60),
    topics={
        "immigration": ["refugees", "border policy", "asylum seekers"],
        "religion": ["religious minorities", "religious practices"],
        "politics": ["elections", "political opponents", "government policy"],
    },
)

## Configuring Languages

`LanguageConfig` maps language codes to names and optionally groups them into tiers.

In [ ]:
lang_config = LanguageConfig(
    languages={
        "en": "English",
        "de": "German",
        "es": "Spanish",
        "ar": "Arabic",
        "hi": "Hindi",
        "sw": "Swahili",
    },
    tiers={
        1: ["en", "de", "es"],      # high-resource: LLMs produce fluent text
        2: ["ar", "hi"],             # medium-resource: quality varies
        3: ["sw"],                   # low-resource: may need pivot strategies
    },
    related_languages={
        "hi": "en",  # for pivot: generate in English, translate to Hindi
        "sw": "en",  # for pivot: generate in English, translate to Swahili
    },
)

print(f"All languages: {lang_config.all_language_codes()}")
print(f"Tier 1: {lang_config.tier_languages(1)}")
print(f"Tier 3: {lang_config.tier_languages(3)}")
print(f"Related language for Swahili: {lang_config.get_related_language('sw')}")

## Direct Generation in Multiple Languages

The simplest approach: ask the LLM to generate text directly in the target language. This works well for languages the LLM was trained on.

In [ ]:
generator = SyntheticDataGenerator(
    task=task,
    llm_provider="openai",
    llm_model="gpt-4o-mini",
    lang_config=lang_config,
)

# Generate for a single language
df_german = generator.generate(
    language="German",
    num_samples=10,
    strategies=["direct"],
    apply_filters=False,
)

print(f"German samples: {len(df_german)}")
for _, row in df_german.head(3).iterrows():
    label = task.label_name(int(row['label']))
    print(f"  [{label}] {row['text'][:80]}...")

## Paraphrasing Existing Data

If you have existing training data, the `"paraphrase"` strategy rewrites each sample with different words while preserving meaning and label.

In [ ]:
import pandas as pd

# Simulating existing Spanish training data
existing_spanish = pd.DataFrame({
    "text": [
        "Los inmigrantes contribuyen enormemente a nuestra economía y cultura.",
        "Cada persona merece respeto independientemente de su origen.",
        "Deberíamos tener un debate abierto sobre las políticas migratorias.",
    ],
    "label": [0, 0, 0],
})

df_paraphrased = generator.generate(
    language="Spanish",
    num_samples=6,
    strategies=["paraphrase"],
    original_df=existing_spanish,
    apply_filters=False,
)

print(f"Paraphrased samples: {len(df_paraphrased)}")
for _, row in df_paraphrased.iterrows():
    print(f"  {row['text'][:90]}...")

## Backtranslation

Backtranslation augments existing data by:
1. Translating the text to an intermediate language (e.g., English)
2. Translating it back to the original language

The round-trip introduces natural paraphrase variation. Requires a translation provider.

In [ ]:
# Backtranslation requires a translation provider.
# Here we show the setup -- uncomment to run if you have Google Cloud Translate configured.

# from synthetictext.providers.google_translate import GoogleTranslateProvider
#
# translator = GoogleTranslateProvider(project_id="my-gcp-project")
#
# generator_with_translation = SyntheticDataGenerator(
#     task=task,
#     llm_provider="openai",
#     translation_provider=translator,
#     lang_config=lang_config,
# )
#
# df_backtrans = generator_with_translation.generate(
#     language="ar",
#     num_samples=20,
#     strategies=["backtranslation"],
#     original_df=existing_arabic_data,
# )

print("Backtranslation requires a translation provider (e.g., Google Cloud Translate).")
print("See the code above for the setup pattern.")

## Pivot Translation

For very low-resource languages where the LLM struggles to generate fluent text directly, the **pivot** strategy:
1. Generates high-quality text in a related high-resource language (e.g., English)
2. Translates the generated text to the target low-resource language

The source language is determined by `LanguageConfig.related_languages`.

In [ ]:
# Pivot also requires a translation provider.

# df_swahili = generator_with_translation.generate(
#     language="sw",
#     num_samples=50,
#     strategies=["pivot"],
# )
# This generates in English (the related language) then translates to Swahili.

print("Pivot translation generates in a high-resource language, then translates.")
print(f"Swahili -> related source: {lang_config.get_related_language('sw')}")
print(f"Hindi -> related source: {lang_config.get_related_language('hi')}")

## Combining Strategies per Language Tier

A practical approach: use different strategies depending on the language's resource level.

In [ ]:
tier_strategies = {
    1: {
        "strategies": ["direct", "paraphrase", "contrastive"],
        "weights": [0.5, 0.3, 0.2],
    },
    2: {
        "strategies": ["direct", "contrastive"],
        "weights": [0.6, 0.4],
    },
    3: {
        # Low-resource: rely more on pivot translation
        "strategies": ["direct", "pivot"],
        "weights": [0.3, 0.7],
    },
}

for tier, config in tier_strategies.items():
    languages = lang_config.tier_languages(tier)
    print(f"Tier {tier} ({languages}):")
    print(f"  Strategies: {config['strategies']}")
    print(f"  Weights: {config['weights']}")
    print()

In [ ]:
# Full loop (uncomment to run -- requires API keys and translation provider for tier 3)

# results = {}
# for tier, config in tier_strategies.items():
#     for lang_code in lang_config.tier_languages(tier):
#         lang_name = lang_config.language_name(lang_code)
#         print(f"\nGenerating for {lang_name} (tier {tier})...")
#         df = generator.generate(
#             language=lang_name,
#             num_samples=100,
#             strategies=config["strategies"],
#             strategy_weights=config["weights"],
#         )
#         results[lang_code] = df
#         print(f"  -> {len(df)} samples")

## Using `generate_all()` for Batch Generation

For simpler cases where you want the same strategy for all languages, `generate_all()` iterates over every language in the config.

In [ ]:
# generate_all returns a dict of {lang_code: DataFrame}

# results = generator.generate_all(
#     num_samples=200,
#     strategies=["direct", "contrastive"],
#     strategy_weights=[0.6, 0.4],
#     original_data_dir="./data/train/",  # optional: loads {lang_code}.csv for paraphrase
#     output_dir="./synthetic_output/",    # auto-saves per-language CSVs
# )

print("generate_all() iterates over all languages in the LanguageConfig.")
print(f"Would generate for: {lang_config.all_language_codes()}")

## Handling Class Imbalance

`LanguageConfig` can store per-language imbalance information, which you can use to adjust generation counts.

In [ ]:
lang_config_imbalanced = LanguageConfig(
    languages={"en": "English", "am": "Amharic"},
    imbalanced={
        "am": {"minority_class": 0, "ratio": 3.0},  # 3:1 imbalance
    },
)

imbalance = lang_config_imbalanced.get_imbalance_info("am")
if imbalance:
    minority = imbalance["minority_class"]
    ratio = imbalance["ratio"]
    print(f"Amharic has {ratio}:1 imbalance")
    print(f"Minority class: {minority} ({task.label_name(minority)})")
    print(f"Consider generating {ratio}x more samples for class {minority}")
else:
    print("No imbalance info")

# English has no imbalance info
print(f"\nEnglish imbalance: {lang_config_imbalanced.get_imbalance_info('en')}")

## Quick Single-Language Config

If you only need one language, use the convenience constructor.

In [ ]:
mono = LanguageConfig.single(code="fr", name="French")
print(f"Languages: {mono.languages}")
print(f"Codes: {mono.all_language_codes()}")

## Summary

| Strategy | Best for | Requires |
|---|---|---|
| `direct` | All languages the LLM handles well | LLM provider |
| `paraphrase` | Augmenting existing labeled data | LLM provider + `original_df` |
| `contrastive` | Sharpening decision boundaries | LLM provider |
| `backtranslation` | Diverse paraphrases via round-trip translation | LLM + translation provider + `original_df` |
| `pivot` | Low-resource languages where direct LLM output is weak | LLM + translation provider + `related_languages` |

**Key takeaways:**
- Use `LanguageConfig` to organize languages into tiers and map codes to names
- Match strategies to resource levels: direct + contrastive for high-resource, pivot for low-resource
- `generate_all()` is a convenience method for batch generation across all configured languages
- Track class imbalance per language and adjust generation ratios accordingly